<a href="https://colab.research.google.com/github/GilangAngga12/Pemrosesan-teks-praktik/blob/main/Datatugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing Teks

In [1]:
 %pip install polars nltk Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.9 MB/s eta 0:00:00


## Library

In [2]:
import re
import html
import polars as pl
import nltk
# import spacy
import unicodedata
from functools import lru_cache
from collections import Counter

# import pandas as pd

from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [3]:
nltk.download("stopwords")

# Model spaCy untuk bahasa Indonesia
# spacy.cli.download("xx_ent_wiki_sm")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Helper Fungsion

In [4]:
def to_list(v):
    """map_elements memberi pl.Series untuk kolom list -> ubah ke list Python."""
    if v is None: return []
    if isinstance(v, pl.Series): return v.to_list()
    return list(v)

def _apply_str(df, col, out, fn):
    return df.with_columns(pl.col(col).map_elements(fn, return_dtype=pl.Utf8).alias(out))

def _apply_list(df, col, out, fn):
    return df.with_columns(
        pl.col(col).map_elements(lambda v: fn(to_list(v)), return_dtype=pl.List(pl.Utf8)).alias(out))

## Dataset

In [8]:
df = pl.read_csv("/content/Bagi DOC-20261018-WA0016.csv")

In [9]:
df

aplikasi,tahun,reviewId,userName,content,score,thumbsUpCount,at,appVersion,reviewCreatedVersion,replyContent,repliedAt,relevant
str,i64,str,str,str,i64,i64,str,str,str,str,str,bool
"""cakap""",2021,"""d66cbb5a-5899-4e4e-8f3b-6c0e0c…","""Budi Ariansyah""","""Aplikasi cerdas ,cepat dan sim…",5,0,"""2021-03-06 14:36:28""","""3.4.3""","""3.4.3""","""Terimakasih atas dukungan kaka…","""2021-03-06 14:57:56""",null
"""ruangguru""",2024,"""9103da38-61a9-417a-b607-dc4aea…","""Finixsa Salsa""","""aplikasi ruang guru sangat mem…",5,0,"""2024-12-27 12:01:22""","""6.93.0""","""6.93.0""","""Makasih ya udah kasih kepercay…","""2024-12-27 12:39:09""",true
"""ruangguru""",2020,"""ce22bae3-c575-4fd3-baa0-2eea35…","""IKhwan617""","""aplikasi nya jelek tapi booong…",5,0,"""2020-11-28 15:03:33""","""6.1.1""","""6.1.1""","""Hai, terima kasih atas review …","""2020-11-28 15:28:50""",true
"""ruangguru""",2024,"""4725fcfd-0c6c-4de5-ba3f-77785f…","""aikoo""","""sebenarnya baru juga download …",2,1,"""2024-07-19 19:24:21""","""6.84.0""","""6.84.0""","""Maaf banget Aikoo kalau nama s…","""2024-07-24 16:27:34""",true
"""pahamify""",2022,"""471da223-1c7f-4821-84e6-7889b2…","""Ghefina Dinda Fauziah""","""baguss sih buat belajar, mudah…",3,3,"""2022-01-17 11:54:21""","""2.0.2""","""2.0.2""","""Hai Ghefina, mohon maaf atas k…","""2022-01-18 11:30:40""",true
…,…,…,…,…,…,…,…,…,…,…,…,…
"""ruangguru""",2021,"""1c210ab7-f8cb-4f32-8148-f0f315…","""Firda Khaerani""","""bagus aplikasinya, banyak lati…",5,0,"""2021-10-11 11:52:25""","""6.24.3""","""6.24.3""","""Hai Firda, terima kasih sudah …","""2021-10-11 12:35:35""",true
"""cakap""",2021,"""6a9e2125-e180-4b48-9dd8-69efd8…","""Agung Setiawan""","""hallo saya Agung Setiawan dari…",5,0,"""2021-09-18 17:05:32""","""3.7.4""","""3.7.4""",null,null,null
"""cakap""",2022,"""44baab89-c59a-458b-8465-2d22eb…","""Eka Novi""","""Bikin belajar bahasa inggris m…",5,0,"""2022-04-04 12:39:56""","""3.9.0""","""3.9.0""","""Terima kasih atas ulasan Kakak…","""2022-04-06 10:53:36""",null


In [ ]:
df.dtypes

[String]

In [15]:
# Muat ulang data asli agar kolom 'content' kembali tersedia
df = pl.read_csv("/content/Bagi DOC-20261018-WA0016.csv")

# Pilih kolom 'content' dan ubah namanya menjadi 'text'
df = df.select(pl.col("content").alias("text"))
df

text
str
"""Aplikasi cerdas ,cepat dan sim…"
"""aplikasi ruang guru sangat mem…"
"""aplikasi nya jelek tapi booong…"
"""sebenarnya baru juga download …"
"""baguss sih buat belajar, mudah…"
…
"""bagus aplikasinya, banyak lati…"
"""hallo saya Agung Setiawan dari…"
"""Bikin belajar bahasa inggris m…"


## Noise Removal

Noise Removal digunakan untuk membersihkan atau menghapus elemen yang biasanya tidak dibutuhkan dalam analisis teks, seperti HTML, URL, alamat email, mention, karakter kontrol, dan spasi berlebih yang tidak relevan dan tidak memiliki nilai informasi untuk analisis teks.

Contoh yang dibersihkan: Tanda baca, emoji, tag HTML, URL, karakter khusus (seperti @, #, $), dan spasi berlebih (whitespace).

### Code

In [16]:
EMOJI = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF\uFE0F\u200d]")

In [17]:
def noise_removal(df, text_col, output_col="text_clean"):
    def clean(text):
        if text is None: return ""
        text = html.unescape(str(text))
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+|\b\w+\.(?:co|com|id)/\S+", " ", text)
        text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)
        text = re.sub(r"(?<!\w)@\w+", " ", text)
        text = re.sub(r"^\s*RT\b", " ", text)
        # hashtag: buang '#', pisahkan CamelCase
        text = re.sub(r"#(\w+)", lambda m: " " + re.sub(r"(?<=[a-z])(?=[A-Z])", " ", m.group(1)) + " ", text)
        text = EMOJI.sub(" ", text)
        text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
        return re.sub(r"\s+", " ", text).strip()
    return _apply_str(df, text_col, output_col, clean)

ROMAN = re.compile(r"^m{0,3}(cm|cd|d?c{0,3})(xc|xl|l?x{0,3})(ix|iv|v?i{0,3})$")
def _collapse(m):
    w = m.group(0)
    if ROMAN.match(w) or w == "www":      # III, XXX, www jangan diubah
        return w
    return re.sub(r"(\w)\1{2,}", r"\1", w)

In [18]:
df_noise = noise_removal(
    df, text_col="text"
)

In [19]:
df_noise

text,text_clean
str,str
"""Aplikasi cerdas ,cepat dan sim…","""Aplikasi cerdas ,cepat dan sim…"
"""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…"
"""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi booong…"
"""sebenarnya baru juga download …","""sebenarnya baru juga download …"
"""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…"
…,…
"""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…"
"""hallo saya Agung Setiawan dari…","""hallo saya Agung Setiawan dari…"
"""Bikin belajar bahasa inggris m…","""Bikin belajar bahasa inggris m…"


In [20]:
print(type(df))

<class 'polars.dataframe.frame.DataFrame'>


## Normalisasi Teks

Normalisasi teks adalah proses menyeragamkan format teks, terutama huruf kapital, Unicode, dan spasi agar memiliki struktur linguistik yang seragam, terutama untuk teks tidak formal, bahasa gaul, singkatan, atau typo (typhographical errors).

Contoh: Mengubah kata "bgt" menjadi "banget", "udh" menjadi "sudah", atau menstandarisasi penulisan angka dan simbol mata uang.

### Code

In [29]:
SLANG = {
    "apk": "aplikasi",
    "app": "aplikasi",
    "apps": "aplikasi",
    "eror": "error",
    "err": "error",
    "bug": "gangguan sistem",
    "lag": "lambat",
    "lemot": "lambat",
    "dwnld": "unduh",
    "download": "unduh",
    "updet": "perbarui",
    "update": "perbarui",
    "rekomended": "direkomendasikan",
    "recomended": "direkomendasikan",
    "recommended": "direkomendasikan",
    "worthit": "sepadan",
    "worth it": "sepadan",
    "fastrespon": "respons cepat",
    "utbk": "ujian tulis berbasis komputer",
    "snbt": "seleksi nasional berbasis tes",
    "snbp": "seleksi nasional berbasis prestasi",
    "tryout": "uji coba",
    "to": "uji coba",
    "mapel": "mata pelajaran",
    "mtk": "matematika",
    "mats": "matematika",
    "bing": "bahasa inggris",
    "bindo": "bahasa indonesia",
    "rg": "ruangguru",
    "pifi": "pahamify",
    "yg": "yang",
    "dg": "dengan",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "karna": "karena",
    "krna": "karena",
    "tdk": "tidak",
    "gak": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "gk": "tidak",
    "enggak": "tidak",
    "engga": "tidak",
    "gaada": "tidak ada",
    "gakada": "tidak ada",
    "bkn": "bukan",
    "sdh": "sudah",
    "udh": "sudah",
    "udah": "sudah",
    "blm": "belum",
    "jg": "juga",
    "jga": "juga",
    "sm": "sama",
    "sma": "sama",
    "klo": "kalau",
    "kalo": "kalau",
    "klu": "kalau",
    "trus": "terus",
    "tp": "tapi",
    "tpi": "tapi",
    "aja": "saja",
    "bgt": "banget",
    "bgtt": "banget",
    "emg": "memang",
    "emang": "memang",
    "org": "orang",
    "dr": "dari",
    "dlm": "dalam",
    "pd": "pada",
    "sbg": "sebagai",
    "spt": "seperti",
    "stlah": "setelah",
    "gw": "saya",
    "gue": "saya",
    "gua": "saya",
    "lu": "kamu",
    "loe": "kamu",
    "sy": "saya",
    "sya": "saya",
    "ak": "aku",
    "bkn": "bukan",
    "dlu": "dulu",
    "dluu": "dulu",
    "makasih": "terima kasih",
    "makasii": "terima kasih",
    "mksih": "terima kasih",
    "trims": "terima kasih",
    "thanks": "terima kasih",
    "thx": "terima kasih",
    "tanks": "terima kasih",
    "tengkyu": "terima kasih",
    "bgs": "bagus",
    "bintang": "bintang",
    "btng": "bintang",
    "bintg": "bintang",
    "kesimpen": "tersimpan",
    "simpen": "simpan",
    "sampe": "sampai",
    "smpe": "sampai",
    "ampe": "sampai",
    "dapet": "dapat",
    "dpt": "dapat",
    "nonton": "menonton",
    "nyesel": "menyesal",
    "moga": "semoga",
    "moga2": "semoga",
    "bikin": "membuat",
    "kasi": "beri",
    "kasih": "beri",
    "gampang": "mudah",
    "mantul": "mantap betul",
    "mantap": "mantap",
    "mantab": "mantap",
    "asik": "asyik",
    "pake": "pakai",
    "pke": "pakai",
    "ngerti": "mengerti",
    "keliatan": "kelihatan",
    "ancur": "hancur",
    "liat": "lihat",
    "boong": "bohong",

}

In [30]:
def text_normalization(df, text_col, output_col="text_normalized", slang=SLANG):
    def norm(text):
        if not text: return ""
        text = unicodedata.normalize("NFKC", text).lower()
        text = re.sub(r"[“”„‟]", '"', text)
        text = re.sub(r"[‘’‚‛]", "'", text)
        text = re.sub(r"[—–]", " - ", text)
        text = re.sub(r"\b\w*(\w)\1{2,}\w*\b", _collapse, text)       # mantaaap -> mantap (angka Romawi dikecualikan)
        text = re.sub(r"\brp\s?(\d)", r"rp \1", text)                # rp76 -> rp 76
        text = re.sub(r"\b([a-z]+)2\b", r"\1-\1", text)             # tiba2 -> tiba-tiba
        text = re.sub(r"\b[a-z0-9]+\b", lambda m: slang.get(m.group(0), m.group(0)), text)
        return re.sub(r"\s+", " ", text).strip()
    return _apply_str(df, text_col, output_col, norm)

In [31]:
df_normalized = text_normalization(
    df_noise, text_col="text_clean"
)

In [32]:
df_normalized

text,text_clean,text_normalized
str,str,str
"""Aplikasi cerdas ,cepat dan sim…","""Aplikasi cerdas ,cepat dan sim…","""aplikasi cerdas ,cepat dan sim…"
"""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…"
"""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi bong h…"
"""sebenarnya baru juga download …","""sebenarnya baru juga download …","""sebenarnya baru juga unduh tap…"
"""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…"
…,…,…
"""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…"
"""hallo saya Agung Setiawan dari…","""hallo saya Agung Setiawan dari…","""hallo saya agung setiawan dari…"
"""Bikin belajar bahasa inggris m…","""Bikin belajar bahasa inggris m…","""membuat belajar bahasa inggris…"


## Tokenisasi

Memecah kalimat atau paragraf yang panjang menjadi unit-unit kecil yang disebut token (biasanya berupa kata atau frasa), hasilnya berupa list, bukan string..

Contoh: Kalimat "Saya sedang belajar NLP." dipecah menjadi: ["Saya", "sedang", "belajar", "NLP"].

### Code

In [33]:
# def tokenization(df, text_col, output_col="tokens"):
#     """
#     Memecah teks menjadi token kata.
#     Output setiap baris berupa list.
#     """
#     result = df.clone()

#     def tokenize_text(text):
#         if text is None:
#             return ""
#         # if pl.isna(text):
#         #     return []

#         text = str(text)

#         # Ambil rangkaian karakter alfanumerik.
#         # Kata dengan apostrof atau tanda hubung dipisah.
#         return re.findall(r"\b\w+\b", text, flags=re.UNICODE)

#     return df.with_columns(
#         df[text_col].map_elements(tokenize_text).alias(output_col)
#     )
#     # result[output_col] = result[text_col].apply(tokenize_text)
#     # return result

In [34]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")
def tokenization(df, text_col, output_col="tokens"):
    return df.with_columns(
        pl.col(text_col).map_elements(lambda t: TOKEN_RE.findall(t or ""),
                                      return_dtype=pl.List(pl.Utf8)).alias(output_col))

In [35]:
df_tokens = tokenization(
    df_normalized, text_col="text_normalized"
)

In [36]:
df_tokens

text,text_clean,text_normalized,tokens
str,str,str,list[str]
"""Aplikasi cerdas ,cepat dan sim…","""Aplikasi cerdas ,cepat dan sim…","""aplikasi cerdas ,cepat dan sim…","[""aplikasi"", ""cerdas"", … ""simple""]"
"""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","[""aplikasi"", ""ruang"", … ""membantu""]"
"""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi bong h…","[""aplikasi"", ""nya"", … ""guru""]"
"""sebenarnya baru juga download …","""sebenarnya baru juga download …","""sebenarnya baru juga unduh tap…","[""sebenarnya"", ""baru"", … ""ya""]"
"""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","[""baguss"", ""sih"", … ""nya""]"
…,…,…,…
"""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","[""bagus"", ""aplikasinya"", … ""soalnya""]"
"""hallo saya Agung Setiawan dari…","""hallo saya Agung Setiawan dari…","""hallo saya agung setiawan dari…","[""hallo"", ""saya"", … ""bangsa""]"
"""Bikin belajar bahasa inggris m…","""Bikin belajar bahasa inggris m…","""membuat belajar bahasa inggris…","[""membuat"", ""belajar"", … ""asyik""]"


## Stopwords Removal

Menghapus kata-kata umum yang sering muncul dalam suatu bahasa namun memiliki sedikit atau bahkan tidak memiliki makna analitik yang kuat.

Contoh: dan, yang, di, dari, ke, untuk

### Code

In [37]:
# Stopword bahasa Indonesia
stop_words = set(stopwords.words("indonesian"))

# Tambahan stopword opsional
custom_stopwords = {
    "yg", "dg", "dgn", "nya", "sih", "dong",
    "nih", "deh", "aja", "udah", "gak", "nggak"
}

stop_words.update(custom_stopwords)

In [38]:
print(stop_words)

{'antara', 'masa', 'tambah', 'bulan', 'menanti', 'berturut-turut', 'bung', 'lanjutnya', 'misalkan', 'dilalui', 'yakin', 'terjadinya', 'menanya', 'diberikannya', 'diperbuat', 'sepihak', 'jadinya', 'tertuju', 'menanyai', 'sedang', 'bagaimana', 'menggunakan', 'berikut', 'berakhirnya', 'artinya', 'bawah', 'dimisalkan', 'menghendaki', 'merekalah', 'saat', 'baru', 'benarkah', 'diperlihatkan', 'gak', 'sekalipun', 'bermacam-macam', 'jauh', 'sekadar', 'penting', 'itulah', 'sebanyak', 'melihatnya', 'sepantasnyalah', 'diketahui', 'teringat-ingat', 'sekitar', 'keadaan', 'selain', 'mengingat', 'agar', 'menunjuknya', 'nih', 'memperbuat', 'dini', 'dijelaskan', 'walaupun', 'panjang', 'menjelaskan', 'ikut', 'sendiri', 'umumnya', 'bersama', 'selanjutnya', 'siap', 'beberapa', 'menambahkan', 'selama-lamanya', 'dari', 'semuanya', 'tadinya', 'bertanya-tanya', 'punya', 'disinilah', 'seterusnya', 'terasa', 'ditanyai', 'jumlahnya', 'apabila', 'tadi', 'terkira', 'kok', 'bagi', 'pertanyaan', 'bagai', 'dikatakan'

In [39]:
def build_stopwords(keep_negation=True):
    sw = set(stopwords.words("indonesian"))
    sw |= {"sih", "dong", "nih", "deh", "kok", "lah", "kan", "nya", "tuh", "nah", "yah", "ya"}
    if keep_negation:
        sw -= {"tidak", "bukan", "tak", "belum", "jangan", "tanpa"}
    return sw

def stopword_removal(df, text_col, output_col="tokens_no_stopwords", stop_words=None):
    stop_words = stop_words or build_stopwords()
    return _apply_list(df, text_col, output_col,
                       lambda toks: [t for t in toks if t not in stop_words and len(t) > 1])

In [40]:
df_no_stopwords = stopword_removal(
    df_tokens, text_col="tokens"
)

In [41]:
df_no_stopwords

text,text_clean,text_normalized,tokens,tokens_no_stopwords
str,str,str,list[str],list[str]
"""Aplikasi cerdas ,cepat dan sim…","""Aplikasi cerdas ,cepat dan sim…","""aplikasi cerdas ,cepat dan sim…","[""aplikasi"", ""cerdas"", … ""simple""]","[""aplikasi"", ""cerdas"", … ""simple""]"
"""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","[""aplikasi"", ""ruang"", … ""membantu""]","[""aplikasi"", ""ruang"", … ""membantu""]"
"""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi bong h…","[""aplikasi"", ""nya"", … ""guru""]","[""aplikasi"", ""jelek"", … ""guru""]"
"""sebenarnya baru juga download …","""sebenarnya baru juga download …","""sebenarnya baru juga unduh tap…","[""sebenarnya"", ""baru"", … ""ya""]","[""unduh"", ""pas"", … ""bintang""]"
"""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","[""baguss"", ""sih"", … ""nya""]","[""baguss"", ""belajar"", … ""kedepan""]"
…,…,…,…,…
"""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","[""bagus"", ""aplikasinya"", … ""soalnya""]","[""bagus"", ""aplikasinya"", ""latihan""]"
"""hallo saya Agung Setiawan dari…","""hallo saya Agung Setiawan dari…","""hallo saya agung setiawan dari…","[""hallo"", ""saya"", … ""bangsa""]","[""hallo"", ""agung"", … ""bangsa""]"
"""Bikin belajar bahasa inggris m…","""Bikin belajar bahasa inggris m…","""membuat belajar bahasa inggris…","[""membuat"", ""belajar"", … ""asyik""]","[""belajar"", ""bahasa"", … ""asyik""]"


In [43]:
df_no_stopwords['tokens_no_stopwords'][6]

""
str
"""bintang"""
"""belom"""
"""mendowload"""


## Stemming

Proses memotong imbuhan (prefiks, sufiks, infiks) pada sebuah kata untuk dikembalikan ke bentuk dasarnya (stem), tanpa selalu memperhatikan kaidah kamus yang valid.

Contoh: Kata "memakan", "dimakan", dan "termakan" akan diubah menjadi akar kata dasar "makan".

### Code

In [44]:
# Inisialisasi stemmer bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

In [45]:
import polars as pl

def to_list(v):
    """Mengubah pl.Series atau iterables lainnya menjadi list Python."""
    if v is None:
        return []
    if isinstance(v, pl.Series):
        return v.to_list()
    if isinstance(v, list):
        return v
    return list(v)

def stemming(df: pl.DataFrame, text_col: str, output_col: str = "tokens_stemmed"):
    """
    Melakukan stemming menggunakan Sastrawi dengan aman dari noise metadata.
    """
    def stem_text(value):
        # Konversi nilai baris (bisa string, list, atau pl.Series) menjadi list token yang bersih
        if value is None:
            tokens = []
        elif isinstance(value, pl.Series):
            tokens = value.to_list()
        elif isinstance(value, list):
            tokens = value
        else:
            # Jika berupa string teks biasa (belum berupa list token)
            tokens = str(value).split()

        # Lakukan stemming pada setiap token
        return [stemmer.stem(str(token)) for token in tokens if token is not None]

    return df.with_columns(
        df[text_col].map_elements(
            stem_text,
            return_dtype=pl.List(pl.Utf8)
        ).alias(output_col)
    )

In [46]:
df_stemmed = stemming(
    df_no_stopwords, text_col="tokens_no_stopwords"
)

In [47]:
df_stemmed

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed
str,str,str,list[str],list[str],list[str]
"""Aplikasi cerdas ,cepat dan sim…","""Aplikasi cerdas ,cepat dan sim…","""aplikasi cerdas ,cepat dan sim…","[""aplikasi"", ""cerdas"", … ""simple""]","[""aplikasi"", ""cerdas"", … ""simple""]","[""aplikasi"", ""cerdas"", … ""simple""]"
"""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","[""aplikasi"", ""ruang"", … ""membantu""]","[""aplikasi"", ""ruang"", … ""membantu""]","[""aplikasi"", ""ruang"", … ""bantu""]"
"""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi bong h…","[""aplikasi"", ""nya"", … ""guru""]","[""aplikasi"", ""jelek"", … ""guru""]","[""aplikasi"", ""jelek"", … ""guru""]"
"""sebenarnya baru juga download …","""sebenarnya baru juga download …","""sebenarnya baru juga unduh tap…","[""sebenarnya"", ""baru"", … ""ya""]","[""unduh"", ""pas"", … ""bintang""]","[""unduh"", ""pas"", … ""bintang""]"
"""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","[""baguss"", ""sih"", … ""nya""]","[""baguss"", ""belajar"", … ""kedepan""]","[""baguss"", ""ajar"", … ""depan""]"
…,…,…,…,…,…
"""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","[""bagus"", ""aplikasinya"", … ""soalnya""]","[""bagus"", ""aplikasinya"", ""latihan""]","[""bagus"", ""aplikasi"", ""latih""]"
"""hallo saya Agung Setiawan dari…","""hallo saya Agung Setiawan dari…","""hallo saya agung setiawan dari…","[""hallo"", ""saya"", … ""bangsa""]","[""hallo"", ""agung"", … ""bangsa""]","[""hallo"", ""agung"", … ""bangsa""]"
"""Bikin belajar bahasa inggris m…","""Bikin belajar bahasa inggris m…","""membuat belajar bahasa inggris…","[""membuat"", ""belajar"", … ""asyik""]","[""belajar"", ""bahasa"", … ""asyik""]","[""ajar"", ""bahasa"", … ""asyik""]"


In [ ]:
# STEM_EXCEPTIONS = {"belasan": "belas"}
# PROTECTED = {"wantimpres", "gibran", "prabowo", "rocky", "gerung", "pratikno", "dpr"}

In [ ]:
# _stemmer = StemmerFactory().create_stemmer()
# @lru_cache(maxsize=None)
# def _stem(tok):
#     if tok in PROTECTED or tok.isdigit() or "-" in tok: return tok
#     if tok in STEM_EXCEPTIONS: return STEM_EXCEPTIONS[tok]
#     return _stemmer.stem(tok)

# def stemming(df, text_col, output_col="tokens_stemmed"):
#     return _apply_list(df, text_col, output_col, lambda toks: [_stem(t) for t in toks])

In [48]:
df_stemmed['tokens_stemmed'][0]

""
str
"""aplikasi"""
"""cerdas"""
"""cepat"""
"""simple"""


## Lemmatization

Proses mengubah kata berimbuhan menjadi bentuk dasarnya berdasarkan kamus dan tata bahasa yang valid (lemma), dengan memperhatikan konteks kalimat. Berbeda dengan stemming, proses ini idealnya menghasilkan bentuk lema yang valid secara linguistik.

Isi kamus lemma sebaiknya ditentukan dari kata berimbuhan yang benar-benar sering muncul dari data, bukan contoh umum. aturan untuk menentukan niai lema bisasnya menggunakan beberaoa ruls seperti:
1. Kalau imbuhan hanya mengubah bentuk gramatikal tanpa mengubah makna inti (dilantik → lantik), pakai kata dasar.

2. Kalau kata berimbuhan sudah punya makna sendiri yang berbeda dari kata dasarnya (pemerintah, kebijakan, anggaran, alasan), pertahankan bentuk aslinya sebagai lemma.

### Cek

### Code

In [50]:

LEMMA_DICTIONARY = {
    "membantu": "bantu",
    "membeli": "beli",
    "membuat": "buat",
    "memahami": "paham",
    "memperbaiki": "baik",
    "mempermudah": "mudah",
    "membalas": "balas",
    "membuka": "buka",
    "melihat": "lihat",
    "menonton": "tonton",
    "menjawab": "jawab",
    "menjelaskan": "jelas",
    "mengetahui": "tahu",
    "mengerti": "erti",
    "menginstal": "instal",
    "mengunduh": "unduh",
    "menggunakan": "guna",
    "mengisi": "isi",
    "menyampaikan": "sampai",
    "menyenangkan": "senang",
    "menambah": "tambah",
    "mendorong": "dorong",
    "meningkatkan": "tingkat",
    "diperbaiki": "baik",
    "dimengerti": "erti",
    "dipahami": "paham",
    "digunakan": "guna",
    "diharapkan": "harap",
    "diberikan": "beri",
    "dibantu": "bantu",
    "dibeli": "beli",
    "diunduh": "unduh",
    "berjalan": "jalan",
    "berfungsi": "fungsi",
    "berkarya": "karya",
    "berkembang": "kembang",
    "bermanfaat": "manfaat",
    "berlangganan": "langgan",
    "terbantu": "bantu",
    "terpakai": "pakai",
    "tersimpan": "simpan",
    "terjadi": "jadi",
    "aplikasinya": "aplikasi",
    "videonya": "video",
    "soalnya": "soal",
    "materinya": "materi",
    "sekolahnya": "sekolah",
    "fiturnya": "fitur",
    "kakaknya": "kakak",
    "belajarmu": "belajar",
    "dukunganmu": "dukung",
    "tambahin": "tambah",
    "kasihlah": "kasih",
    "berikan": "beri",
    "masukkan": "masuk",
    "pahami": "paham",
    "perkembangan": "kembang",
    "pembelajaran": "ajar",
    "pelajaran": "ajar",
    "pengajaran": "ajar",
    "ketidaknyamanan": "nyaman",
    "kenyamanan": "nyaman",
    "kepercayaan": "percaya",
    "penyemangat": "semangat",
    "bimbingan": "bimbing",
    "jawaban": "jawab",
    "penjelasan": "jelas",
    "sebulannya": "bulan",
}

In [51]:

def lemmatization(
    df: pl.DataFrame,
    text_col: str,
    output_col: str = "tokens_lemmatized",
    lemma_dict: dict = None
):
    """
    Lemmatization berbasis kamus.
    Kata yang tidak ada dalam kamus dipertahankan.
    """
    if lemma_dict is None:
        lemma_dict = lemma_dictionary

    def lemmatize_text(value):
        if value is None:
            return []
        # kolom list di Polars masuk sebagai pl.Series -> ubah ke list Python
        if isinstance(value, pl.Series):
            tokens = value.to_list()
        elif isinstance(value, list):
            tokens = value
        else:
            tokens = str(value).split()

        return [lemma_dict.get(str(t).lower(), str(t).lower()) for t in tokens]

    return df.with_columns(
        pl.col(text_col).map_elements(
            lemmatize_text,
            return_dtype=pl.List(pl.Utf8)
        ).alias(output_col)
    )

In [52]:
df_lemmatized = lemmatization(
    df_no_stopwords, text_col="tokens_no_stopwords"
)

In [53]:
df_lemmatized

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_lemmatized
str,str,str,list[str],list[str],list[str]
"""Aplikasi cerdas ,cepat dan sim…","""Aplikasi cerdas ,cepat dan sim…","""aplikasi cerdas ,cepat dan sim…","[""aplikasi"", ""cerdas"", … ""simple""]","[""aplikasi"", ""cerdas"", … ""simple""]","[""aplikasi"", ""cerdas"", … ""simple""]"
"""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","[""aplikasi"", ""ruang"", … ""membantu""]","[""aplikasi"", ""ruang"", … ""membantu""]","[""aplikasi"", ""ruang"", … ""membantu""]"
"""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi bong h…","[""aplikasi"", ""nya"", … ""guru""]","[""aplikasi"", ""jelek"", … ""guru""]","[""aplikasi"", ""jelek"", … ""guru""]"
"""sebenarnya baru juga download …","""sebenarnya baru juga download …","""sebenarnya baru juga unduh tap…","[""sebenarnya"", ""baru"", … ""ya""]","[""unduh"", ""pas"", … ""bintang""]","[""unduh"", ""pas"", … ""bintang""]"
"""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","[""baguss"", ""sih"", … ""nya""]","[""baguss"", ""belajar"", … ""kedepan""]","[""baguss"", ""belajar"", … ""kedepan""]"
…,…,…,…,…,…
"""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","[""bagus"", ""aplikasinya"", … ""soalnya""]","[""bagus"", ""aplikasinya"", ""latihan""]","[""bagus"", ""aplikasinya"", ""latihan""]"
"""hallo saya Agung Setiawan dari…","""hallo saya Agung Setiawan dari…","""hallo saya agung setiawan dari…","[""hallo"", ""saya"", … ""bangsa""]","[""hallo"", ""agung"", … ""bangsa""]","[""hallo"", ""agung"", … ""bangsa""]"
"""Bikin belajar bahasa inggris m…","""Bikin belajar bahasa inggris m…","""membuat belajar bahasa inggris…","[""membuat"", ""belajar"", … ""asyik""]","[""belajar"", ""bahasa"", … ""asyik""]","[""belajar"", ""bahasa"", … ""asyik""]"


## Hasil

In [54]:
columns_to_view = [
    "text",
    "text_clean",
    "text_normalized",
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized"
]

df_comparison = (
    df_lemmatized
    .with_columns(df_stemmed["tokens_stemmed"])
    .select(columns_to_view)
)

df_comparison.head()

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized
str,str,str,list[str],list[str],list[str],list[str]
"""Aplikasi cerdas ,cepat dan sim…","""Aplikasi cerdas ,cepat dan sim…","""aplikasi cerdas ,cepat dan sim…","[""aplikasi"", ""cerdas"", … ""simple""]","[""aplikasi"", ""cerdas"", … ""simple""]","[""aplikasi"", ""cerdas"", … ""simple""]","[""aplikasi"", ""cerdas"", … ""simple""]"
"""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","[""aplikasi"", ""ruang"", … ""membantu""]","[""aplikasi"", ""ruang"", … ""membantu""]","[""aplikasi"", ""ruang"", … ""bantu""]","[""aplikasi"", ""ruang"", … ""membantu""]"
"""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi bong h…","[""aplikasi"", ""nya"", … ""guru""]","[""aplikasi"", ""jelek"", … ""guru""]","[""aplikasi"", ""jelek"", … ""guru""]","[""aplikasi"", ""jelek"", … ""guru""]"
"""sebenarnya baru juga download …","""sebenarnya baru juga download …","""sebenarnya baru juga unduh tap…","[""sebenarnya"", ""baru"", … ""ya""]","[""unduh"", ""pas"", … ""bintang""]","[""unduh"", ""pas"", … ""bintang""]","[""unduh"", ""pas"", … ""bintang""]"
"""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","[""baguss"", ""sih"", … ""nya""]","[""baguss"", ""belajar"", … ""kedepan""]","[""baguss"", ""ajar"", … ""depan""]","[""baguss"", ""belajar"", … ""kedepan""]"


In [55]:
df_export = df_comparison.clone()

token_columns = [
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized"
]

# Ubah list token menjadi string
for col in token_columns:
    df_export = df_export.with_columns(
        df_export[col].map_elements(
            lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens),
            return_dtype=pl.Utf8
        ).alias(col)
    )

# # Simpan hasil ke CSV
# df_export.write_csv(
#     "hasil_preprocessing.csv",
#     encoding="utf8"   # Polars tidak pakai "utf-8-sig", cukup "utf8"
# )

In [57]:
df_export = df_comparison.with_columns(
    [pl.col(c).list.join(" ") for c in token_columns]
)

df_export.write_csv("hasil_preprocessingdatasample.csv")

In [58]:
df_export.write_csv("hasil_preprocessingdatasample.csv")

In [59]:
df_load = pl.read_csv('hasil_preprocessingdatasample.csv')

In [60]:
df_load

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized
str,str,str,str,str,str,str
"""Aplikasi cerdas ,cepat dan sim…","""Aplikasi cerdas ,cepat dan sim…","""aplikasi cerdas ,cepat dan sim…","""aplikasi cerdas cepat dan simp…","""aplikasi cerdas cepat simple""","""aplikasi cerdas cepat simple""","""aplikasi cerdas cepat simple"""
"""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru sangat mem…","""aplikasi ruang guru membantu""","""aplikasi ruang guru bantu""","""aplikasi ruang guru membantu"""
"""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi booong…","""aplikasi nya jelek tapi bong h…","""aplikasi nya jelek tapi bong h…","""aplikasi jelek bong hayuk teri…","""aplikasi jelek bong hayuk teri…","""aplikasi jelek bong hayuk teri…"
"""sebenarnya baru juga download …","""sebenarnya baru juga download …","""sebenarnya baru juga unduh tap…","""sebenarnya baru juga unduh tap…","""unduh pas menonton video coc n…","""unduh pas tonton video coc nam…","""unduh pas nonton video coc nam…"
"""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","""baguss sih buat belajar, mudah…","""baguss sih buat belajar mudah …","""baguss belajar mudah pahami su…","""baguss ajar mudah paham suka e…","""baguss belajar mudah pahami su…"
…,…,…,…,…,…,…
"""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","""bagus aplikasinya, banyak lati…","""bagus aplikasinya banyak latih…","""bagus aplikasinya latihan""","""bagus aplikasi latih""","""bagus aplikasinya latihan"""
"""hallo saya Agung Setiawan dari…","""hallo saya Agung Setiawan dari…","""hallo saya agung setiawan dari…","""hallo saya agung setiawan dari…","""hallo agung setiawan prakerja …","""hallo agung setiawan prakerja …","""hallo agung setiawan prakerja …"
"""Bikin belajar bahasa inggris m…","""Bikin belajar bahasa inggris m…","""membuat belajar bahasa inggris…","""membuat belajar bahasa inggris…","""belajar bahasa inggris asyik""","""ajar bahasa inggris asyik""","""belajar bahasa inggris asyik"""
